Importing Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFECV
from sklearn.tree import DecisionTreeClassifier

from statsmodels.stats.outliers_influence import variance_inflation_factor
import math

Importing Dataset

In [2]:
steph = pd.read_excel("Steph_curry.xlsx")

Creating Deep Copy of dataset.

In [3]:
new_steph = steph.copy()

Changing Date to Datetime data type.

Sorting dataset by dates.

In [4]:
new_steph['Date'] = pd.to_datetime(steph['Date'])

new_steph = new_steph.sort_values("Date")

Creating target Variable, being Over_Under_25

In [5]:
new_steph["Over_under_25"] = (new_steph["PTS"] > 24.8).astype(int)

Removing Rows missing opponent or date.
Removing column as unnessesary. 

In [6]:
new_steph = new_steph.dropna(subset=["Opp", "Date"])
new_steph = new_steph.drop(columns=["Unnamed: 5", "Rk", "Gcar", "Gtm"])

Numerical values missing changes to mean

In [7]:
cols = ['FT', 'TOV', 'GmSc', '+/-', 'FG', 'FGA', '3P', '3PA', '2P', '2PA', 'FTA', 'ORB', 'DRB', 'TRB', 'AST','STL','BLK','PF']


new_steph[cols] = new_steph[cols].fillna(
    new_steph[cols].mean()
)
new_steph['FT'].isna().sum()

np.int64(0)

Percentage Values changed to median

In [8]:
cols = ['FG%', 'eFG%', '3P%', '2P%', 'FT%']

new_steph[cols] = new_steph[cols].fillna(
    new_steph[cols].median()
)
new_steph['eFG%'].isna().sum()

np.int64(0)

Creating historical Averages of statistics, based on previous 5 games against each team. 

First game against each team can be removed from dataset, as they wont have historical games to work with. 

In [9]:
num_cols = [
    "FG", "FGA", "3P", "3PA",
    "FT", "FTA", "TRB", "AST", "STL", "TOV", "FG%", "2P%", "eFG%", "FT%",
    "3P%", "2P", "2PA", "PF"
]

for col in num_cols:
    new_steph[f"{col}5_vs_opp"] = (
        new_steph.groupby("Opp")[col]
        .transform(lambda x: x.shift().rolling(5, min_periods=1).mean())
    )
historical_features = [f"{col}5_vs_opp" for col in num_cols]
new_steph = new_steph.dropna(subset=historical_features).copy()

output_cols = ["Date", "Opp"] + [f"{col}5_vs_opp" for col in num_cols]

display(new_steph[output_cols].head(20))

,Date,Opp,FG5_vs_opp,FGA5_vs_opp,3P5_vs_opp,3PA5_vs_opp,FT5_vs_opp,FTA5_vs_opp,TRB5_vs_opp,AST5_vs_opp,STL5_vs_opp,TOV5_vs_opp,FG%5_vs_opp,2P%5_vs_opp,eFG%5_vs_opp,FT%5_vs_opp,3P%5_vs_opp,2P5_vs_opp,2PA5_vs_opp,PF5_vs_opp
15,2009-11-30,IND,3.0,7.0,0.0,2.0,0.0,0.0,1.0,4.0,1.0,3.0,0.4290,0.6000,0.4290,1.00,0.0000,3.0,5.0,2.0
17,2009-12-03,HOU,7.0,12.0,0.0,1.0,0.0,0.0,2.0,7.0,4.0,2.0,0.5830,0.6360,0.5830,1.00,0.0000,7.0,11.0,2.0
24,2009-12-16,SAS,5.0,16.0,1.0,4.0,0.0,0.0,7.0,5.0,1.0,6.0,0.3130,0.3330,0.3440,1.00,0.2500,4.0,12.0,3.0
26,2009-12-22,MEM,3.0,6.0,1.0,2.0,0.0,0.0,5.0,9.0,2.0,1.0,0.5000,0.5000,0.5830,1.00,0.5000,2.0,4.0,4.0
28,2009-12-26,PHO,5.0,9.0,2.0,3.0,0.0,0.0,2.0,4.0,1.0,3.0,0.5560,0.5000,0.6670,1.00,0.6670,3.0,6.0,5.0
29,2009-12-28,BOS,4.0,10.0,1.0,4.0,4.0,4.0,0.0,7.0,4.0,6.0,0.4000,0.5000,0.4500,1.00,0.2500,3.0,6.0,3.0
30,2009-12-29,LAL,5.0,15.0,1.0,5.0,2.0,2.0,4.0,8.0,0.0,1.0,0.3330,0.4000,0.3670,1.00,0.2000,4.0,10.0,0.0
31,2010-01-02,POR,3.0,9.0,2.0,3.0,4.0,5.0,6.0,8.0,1.0,1.0,0.3330,0.1670,0.4440,0.80,0.6670,1.0,6.0,6.0
32,2010-01-05,DEN,4.0,14.0,2.0,6.0,0.0,0.0,4.0,6.0,2.0,1.0,0.2860,0.2500,0.3570,1.00,0.3330,2.0,8.0,2.0
33,2010-01-06,MIN,2.0,8.0,1.0,2.0,3.0,4.0,2.0,5.0,1.0,2.0,0.2500,0.1670,0.3130,0.75,0.5000,1.0,6.0,1.0


One Hot Encoding

In [10]:
encoded_steph = pd.get_dummies(
    new_steph,
    columns=['Opp'],
    drop_first=True
)

print("One-Hot Encoded Data:")
print(encoded_steph.head())

One-Hot Encoded Data:
         Date Team      Result GS               MP   FG   FGA    FG%   3P  \
15 2009-11-30  GSW  W, 126-107  *         19:50:00  2.0   5.0  0.400  1.0   
17 2009-12-03  GSW  L, 109-111  *   1 day, 7:51:00  4.0   8.0  0.500  1.0   
24 2009-12-16  GSW   L, 91-103  *   1 day, 1:03:00  4.0  11.0  0.364  0.0   
26 2009-12-22  GSW  L, 108-121  *  1 day, 17:17:00  6.0  15.0  0.400  2.0   
28 2009-12-26  GSW  W, 132-127  *   1 day, 2:33:00  4.0  10.0  0.400  3.0   

    3PA  ...  Opp_OKC  Opp_ORL  Opp_PHI  Opp_PHO  Opp_POR  Opp_SAC  Opp_SAS  \
15  2.0  ...    False    False    False    False    False    False    False   
17  3.0  ...    False    False    False    False    False    False    False   
24  2.0  ...    False    False    False    False    False    False     True   
26  6.0  ...    False    False    False    False    False    False    False   
28  4.0  ...    False    False    False     True    False    False    False   

    Opp_TOR  Opp_UTA  Opp_WAS  
15    Fa

Train, Validate Test Set:

Cutoff: 0.7 cuts off the last 30% of rows,
        0.85 cuts off the last 15% of rows. 

Iloc: allows us to select where from the data is(:set means before the cutoff) (set: means anything after the cutoff.)

In [11]:
X = new_steph[historical_features + ["Opp"]]
y = new_steph['Over_under_25']

train_cutoff = int(len(new_steph) * 0.70)
val_cutoff = int(len(new_steph) * 0.85)

X_train = X.iloc[:train_cutoff]
y_train = y.iloc[:train_cutoff]

X_val = X.iloc[train_cutoff:val_cutoff]
y_val = y.iloc[train_cutoff:val_cutoff]

X_test = X.iloc[val_cutoff:]
y_test = y.iloc[val_cutoff:]

Scaling

In [12]:
exclude = [ "PTS", "Over_under_25"]
pca_cols = [col for col in num_cols if col not in exclude]

scaler = StandardScaler()
standardized_data = scaler.fit_transform(new_steph[pca_cols])

new_steph_standardized = pd.DataFrame(
    standardized_data,
    columns=pca_cols,
    index=new_steph.index
)

In [13]:

print("Standardised means:")
print(new_steph_standardized.mean().round(2))

print("\nStandardised standard deviations:")
print(new_steph_standardized.std(ddof=0).round(2))

Standardised means:
FG     -0.0
FGA     0.0
3P      0.0
3PA     0.0
FT     -0.0
FTA    -0.0
TRB     0.0
AST     0.0
STL    -0.0
TOV     0.0
FG%     0.0
2P%    -0.0
eFG%   -0.0
FT%    -0.0
3P%    -0.0
2P     -0.0
2PA     0.0
PF      0.0
dtype: float64

Standardised standard deviations:
FG      1.0
FGA     1.0
3P      1.0
3PA     1.0
FT      1.0
FTA     1.0
TRB     1.0
AST     1.0
STL     1.0
TOV     1.0
FG%     1.0
2P%     1.0
eFG%    1.0
FT%     1.0
3P%     1.0
2P      1.0
2PA     1.0
PF      1.0
dtype: float64


PCA

In [14]:
pca = PCA()
pca_result = pca.fit_transform(standardized_data)

explained_varience = pca.explained_variance_ratio_
cumulative_varience = np.cumsum(explained_varience)

pca_summary = pd.DataFrame({
    "Principal Component": [
        f"PC{i+1}" for i in range(len(explained_varience))
    ],
    "Explained Variance": explained_varience,
    "Cumulative Variance": cumulative_varience
})

print(pca_summary.round(3))

   Principal Component  Explained Variance  Cumulative Variance
0                  PC1               0.273                0.273
1                  PC2               0.139                0.412
2                  PC3               0.109                0.520
3                  PC4               0.094                0.614
4                  PC5               0.071                0.686
5                  PC6               0.058                0.744
6                  PC7               0.056                0.799
7                  PC8               0.054                0.853
8                  PC9               0.047                0.900
9                 PC10               0.046                0.946
10                PC11               0.043                0.989
11                PC12               0.005                0.994
12                PC13               0.003                0.997
13                PC14               0.002                0.999
14                PC15               0.0

In [15]:
pca_dataset = pd.DataFrame(pca_result)
print(pca_dataset.head())

         0         1         2         3         4         5         6   \
0 -4.651964  2.642889 -0.161348  0.972435  0.756166 -0.104326 -0.773242   
1 -2.939730  2.013151  1.001773  1.265335  0.301198 -0.887137  0.763818   
2 -4.518350 -0.372182  2.660111  0.120821  0.549892 -0.687987  0.062979   
3 -2.256906  0.260820  1.035805 -1.192415  0.265770  0.128185  1.274632   
4 -3.074996  2.375631 -0.673121 -0.136010  1.744404 -1.413308 -1.235956   

         7         8         9         10        11        12        13  \
0  0.936444 -0.375088 -0.933729  1.033586  0.077023  0.247489  0.300465   
1 -0.969255 -0.320292 -0.706850  1.369537 -0.264860 -0.088250 -0.124253   
2  0.484784  1.674160 -0.687009  2.589791 -1.122502  0.035596 -0.543867   
3 -0.638661 -1.087528  1.265406 -0.316467 -0.082505  0.020971 -0.100977   
4  2.075854  0.189962  0.167372  1.497440  0.531290  0.177995  0.398576   

         14        15            16            17  
0  0.005472 -0.004948  1.061950e-14  7.833255e

In [16]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=pca_cols,
    columns=[f"PC{i+1}" for i in range(len(pca_cols))]
)

print(loadings.iloc[:, :9].round(2))

       PC1   PC2   PC3   PC4   PC5   PC6   PC7   PC8   PC9
FG    0.44 -0.05  0.07 -0.13 -0.06 -0.02  0.00  0.05  0.02
FGA   0.32 -0.36 -0.04 -0.29 -0.07 -0.02  0.04  0.08  0.08
3P    0.34  0.13 -0.36 -0.24  0.05 -0.01 -0.04  0.05  0.10
3PA   0.26 -0.17 -0.37 -0.28 -0.12  0.16  0.13 -0.13  0.32
FT    0.16 -0.25 -0.28  0.56  0.00 -0.03 -0.03  0.12 -0.03
FTA   0.16 -0.27 -0.28  0.54 -0.02 -0.05 -0.16  0.03 -0.02
TRB   0.13 -0.18 -0.10 -0.02  0.21  0.15  0.20 -0.40 -0.30
AST   0.02 -0.08  0.13 -0.02  0.41  0.59 -0.09  0.06  0.13
STL   0.03 -0.09  0.12  0.05  0.48  0.32 -0.32  0.11  0.12
TOV   0.06 -0.08 -0.01 -0.01  0.45 -0.23  0.27 -0.50 -0.32
FG%   0.33  0.36  0.18  0.16  0.01  0.01 -0.02 -0.03 -0.03
2P%   0.22  0.15  0.31  0.28 -0.26  0.24  0.20 -0.35  0.25
eFG%  0.34  0.40  0.01  0.09  0.05  0.01 -0.04 -0.02 -0.01
FT%  -0.00  0.07 -0.04  0.11  0.15  0.13  0.80  0.53 -0.03
3P%   0.24  0.37 -0.13 -0.05  0.25 -0.20 -0.19  0.24 -0.26
2P    0.28 -0.20  0.48  0.06 -0.15 -0.02  0.05  0.03 -0.

### CA2 Starting Point. 

#### 1. Feature Engineering

* Over_under_25 = Feature in which is our target variable. Binary value, with 1 meaning over 25 points and 0 meaning under 25. 

* ()5_vs_opp = 18 Features created to track average of each value of each statistic over the previous 5 games. 5 games were chosen as form, team dependancy and consistency suit 5 games well. 


#### 2. Modelling

Data has previously been split into a Train, Validation and Test set.